# Training Notebook for OMR Assembler Model
Step-by-step training workflow for music notation assembly

## 1. Setup and Imports

In [29]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import os
import glob
import yaml

from utils.data_pool import load_munglinker_data
from mung.io import read_nodes_from_file
from utils.constants import get_classlist_and_classdict
from utils.metrics import compute_matching_score
from utils.utility import set_seed
from configs.assembler.default import get_cfg_defaults
from model.model import MLP, MLPwithSoftClass, MLPwithSoftClassExtraMLP

## 2. Configuration

In [35]:
# Experiment configuration
exp_name = 'first_training'
model_config_path = 'configs/assembler/MLP32_nogrammar.yaml'
output_dir = 'outputs'

# Data paths
mung_root = 'data/MUSCIMA++/v2.0/data/annotations'
gt_mung_root = 'data/MUSCIMA++/v2.0/data/annotations'
images_root = 'data/MUSCIMA++/datasets_r_staff/images'
split_file = 'splits/mob_split.yaml'
classes = 'essential'

# Training settings
load_epochs = 0  # Set to > 0 to resume from checkpoint
test_only = False
val_only = False

# Device - prioritize MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'Using device: {device} (Apple Metal Performance Shaders)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using device: {device} (NVIDIA CUDA)')
else:
    device = torch.device('cpu')
    print(f'Using device: {device} (CPU)')
print(f'PyTorch version: {torch.__version__}')

Using device: mps (Apple Metal Performance Shaders)
PyTorch version: 2.8.0


## 3. Load Configuration

In [31]:
# Load model config
cfg = get_cfg_defaults()
cfg.merge_from_file(model_config_path)

# Set seed for reproducibility
set_seed(cfg.SYSTEM.SEED)

# Create output directory
os.makedirs(f'{output_dir}/{exp_name}', exist_ok=True)

print('Configuration loaded!')
print(f'Number of epochs: {cfg.TRAIN.NUM_EPOCHS}')
print(f'Batch size: {cfg.TRAIN.BATCH_SIZE}')
print(f'Learning rate: {cfg.TRAIN.LEARNING_RATE}')
print(f'POS_WEIGHT: {cfg.TRAIN.POS_WEIGHT}')
print(f'Model mode: {cfg.MODEL.MODE}')

Configuration loaded!
Number of epochs: 200
Batch size: 256
Learning rate: 0.001
POS_WEIGHT: 1
Model mode: MLP


## 4. Load Data

In [32]:
# Get class information
class_list, class_dict = get_classlist_and_classdict(classes)
print(f'Loaded {len(class_list)} classes')

# Load data configuration
with open(cfg.DATA.DATA_CONFIG, 'rb') as hdl:
    data_config = yaml.load(hdl, Loader=yaml.FullLoader)
data_config['mode'] = cfg.MODEL.MODE

print('\nLoading data...')
data = load_munglinker_data(
    mung_root=mung_root,
    images_root=images_root,
    split_file=split_file,
    class_list=class_list,
    class_dict=class_dict,
    config=data_config,
    load_training_data=True,
    load_validation_data=True,
    load_test_data=False,
)

print(f'\nTraining samples: {len(data["train"]):,}')
print(f'Validation samples: {len(data["valid"]):,}')

Loaded 73 classes

Loading data...
Loading training data...


Loading MuNG-pairs: 100%|██████████| 84/84 [00:33<00:00,  2.47it/s]


Loading validation data...


Loading MuNG-pairs: 100%|██████████| 28/28 [00:09<00:00,  2.85it/s]


Training samples: 1,668,285
Validation samples: 11,111,037


## 5. Build Model

In [36]:
# Build model based on config
if cfg.MODEL.MODE == "MLP":
    model = MLP(cfg)
elif cfg.MODEL.MODE == "MLPwithSoftClass":
    model = MLPwithSoftClass(cfg)
elif cfg.MODEL.MODE == "MLPwithSoftClassExtraMLP":
    model = MLPwithSoftClassExtraMLP(cfg)
else:
    raise ValueError(f"Model {cfg.MODEL.MODE} is not supported")

model = model.to(device)

print('Model built!')
print(f'Model type: {cfg.MODEL.MODE}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

Model built!
Model type: MLP
Total parameters: 9,089


## 6. Setup Optimizer and Loss

In [38]:
# Save config
with open(f"{output_dir}/{exp_name}/config.yaml", 'w') as f:
    f.write(cfg.dump())

# Setup optimizer

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.TRAIN.LEARNING_RATE)

# Setup loss function
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(cfg.TRAIN.POS_WEIGHT))

print('Optimizer and loss function setup complete!')
print(f'Optimizer: {cfg.TRAIN.OPTIMIZER}')
print(f'Learning rate: {cfg.TRAIN.LEARNING_RATE}')
print(f'POS_WEIGHT: {cfg.TRAIN.POS_WEIGHT}')

Optimizer and loss function setup complete!
Optimizer: AdamW
Learning rate: 0.001
POS_WEIGHT: 1


## 7. Create DataLoader

In [39]:
# Create training data loader
train_loader = DataLoader(
    data['train'], 
    batch_size=cfg.TRAIN.BATCH_SIZE, 
    shuffle=True, 
    num_workers=cfg.SYSTEM.NUM_WORKERS
)

print(f'Training DataLoader created!')
print(f'Batch size: {cfg.TRAIN.BATCH_SIZE}')
print(f'Number of batches: {len(train_loader)}')
print(f'Num workers: {cfg.SYSTEM.NUM_WORKERS}')

Training DataLoader created!
Batch size: 256
Number of batches: 6517
Num workers: 0


## 8. Load Checkpoint (Optional)
If resuming training, load checkpoint here

In [42]:
# Load checkpoint if specified
if load_epochs > 0:
    checkpoint_path = f"{output_dir}/{exp_name}/model_ep{load_epochs}.pth"
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    print(f"Loaded model at epoch {checkpoint['epoch']}")
    
    # Skip random states to maintain reproducibility
    for _ in tqdm(range(load_epochs), desc="Skipping Epochs"):
        torch.empty((), dtype=torch.int64).random_()
        torch.empty((), dtype=torch.int64).random_()
else:
    print('Starting training from scratch')

Starting training from scratch


## 9. Training Loop

In [43]:
# Training loop
model.train()
best_f1 = 0.0

for epoch in range(load_epochs, cfg.TRAIN.NUM_EPOCHS):
    corr = 0
    total = 0
    epoch_loss = 0.0
    num_batches = 0
    
    print(f'\n=== Epoch {epoch+1}/{cfg.TRAIN.NUM_EPOCHS} ===')
    
    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch+1}'):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass
        optimizer.zero_grad()
        output = model(batch)
        loss = criterion(output, batch['label'])
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        pred = torch.sigmoid(output) > 0.5
        corr += (pred == batch['label']).sum().item()
        total += len(batch['label'])
        epoch_loss += loss.item()
        num_batches += 1
    
    # Print epoch statistics
    avg_loss = epoch_loss / num_batches
    accuracy = corr / total
    print(f'Epoch {epoch+1} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')
    
    # Save checkpoint at specified frequency
    if (epoch+1) % cfg.TRAIN.SAVE_FREQUENCY == 0 and (epoch+1) != cfg.TRAIN.NUM_EPOCHS:
        checkpoint_path = f'{output_dir}/{exp_name}/model_ep{epoch+1}.pth'
        checkpoint = {
            'epoch': epoch + 1,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict()
        }
        torch.save(checkpoint, checkpoint_path)
        print(f'Checkpoint saved to {checkpoint_path}')

print('\n=== Training Complete ===')


=== Epoch 1/200 ===


Training Epoch 1:   0%|          | 0/6517 [00:00<?, ?it/s]

Epoch 1 - Loss: 0.1449, Accuracy: 0.9430

=== Epoch 2/200 ===


Training Epoch 2:   0%|          | 0/6517 [00:00<?, ?it/s]

Epoch 2 - Loss: 0.1190, Accuracy: 0.9480

=== Epoch 3/200 ===


...

Epoch 198 - Loss: 0.0177, Accuracy: 0.9930

=== Epoch 199/200 ===


Training Epoch 199:   0%|          | 0/6517 [00:00<?, ?it/s]

Epoch 199 - Loss: 0.0177, Accuracy: 0.9929

=== Epoch 200/200 ===


Training Epoch 200:   0%|          | 0/6517 [00:00<?, ?it/s]

Epoch 200 - Loss: 0.0177, Accuracy: 0.9929

=== Training Complete ===


## 10. Save Final Model

In [44]:
# Save final model
final_checkpoint_path = f'{output_dir}/{exp_name}/model_final.pth'
checkpoint = {
    'epoch': cfg.TRAIN.NUM_EPOCHS,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict()
}
torch.save(checkpoint, final_checkpoint_path)
print(f'Final model saved to {final_checkpoint_path}')

Final model saved to outputs/first_training/model_final.pth


## 11. Validation
Run validation on trained model

In [45]:
# Prepare validation
model.eval()
all_mung_files = glob.glob(mung_root + "/**/*.xml", recursive=True)
all_gt_files = glob.glob(gt_mung_root + "/**/*.xml", recursive=True)

with open(split_file, 'rb') as hdl:
    split = yaml.load(hdl, Loader=yaml.FullLoader)

include_names = split['valid']
valid_data = data['valid']

mung_files_in_split = sorted([f for f in all_mung_files if os.path.splitext(os.path.basename(f))[0] in include_names])
gt_files_in_split = sorted([f for f in all_gt_files if os.path.splitext(os.path.basename(f))[0] in include_names])

print(f'Validation files: {len(mung_files_in_split)}')

Validation files: 28


In [46]:
# Check for soft class probabilities
class_prob_files = glob.glob(mung_root + "/**/*.npy", recursive=True)
class_prob_files = sorted([f for f in class_prob_files if os.path.splitext(os.path.basename(f))[0] in include_names])
USE_HARD_LABEL = len(class_prob_files) == 0

if USE_HARD_LABEL:
    print("No soft label found. Using top-1 hard labels instead.")
else:
    print(f"Using soft class probabilities from {len(class_prob_files)} files")

No soft label found. Using top-1 hard labels instead.


In [ ]:
# Get inference graph
inference_graph = valid_data.get_inference_graph() if isinstance(valid_data.inference_graph[0], list) else valid_data.inference_graph
total_matching_score = np.array([0.0, 0.0, 0.0, 0.0])

print('Running validation inference...')
for i in tqdm(range(len(mung_files_in_split)), desc="Validation"):
    mung_file = mung_files_in_split[i]
    gt_file = gt_files_in_split[i]
    node_list = read_nodes_from_file(mung_file)
    gt_list = read_nodes_from_file(gt_file)
    
    # Prepare class probabilities
    if USE_HARD_LABEL:
        class_label = np.array([class_dict[node.class_name] for node in node_list])
        class_prob = np.zeros((len(node_list), cfg.MODEL.VOCAB_DIM))
        class_prob[np.arange(len(class_label)), class_label] = 1
    else:
        class_prob = np.load(class_prob_files[i])
    
    edge_list = []
    cur_graph = inference_graph[i]
    
    # Run inference on all pairs
    with torch.no_grad():
        for batch_idx in range((cur_graph['source_id'].shape[0] // cfg.EVAL.BATCH_SIZE) + 1):
            start_idx = batch_idx * cfg.EVAL.BATCH_SIZE
            end_idx = start_idx + cfg.EVAL.BATCH_SIZE
            
            batch = {k: v[start_idx:end_idx].to(device) for k, v in cur_graph.items()}
            
            if batch['source_id'].shape[0] == 0:
                continue
                
            output = model(batch)
            output = torch.sigmoid(output)
            
            for idx in range(batch['source_id'].shape[0]):
                source_id = batch['source_id'][idx]
                target_id = batch['target_id'][idx]
                edge_list.append((source_id.item(), target_id.item(), output[idx].item()))
    
    # Compute matching score for this file
    matching_score = compute_matching_score(node_list, class_prob, edge_list, gt_list, class_list, class_dict)
    total_matching_score += matching_score

# Print final validation results
avg_scores = total_matching_score / len(mung_files_in_split)
print(f'\n=== Validation Results ===')
print(f'Average AUC: {avg_scores[0]:.4f}')
print(f'Average F1: {avg_scores[1]:.4f}')
print(f'Average Precision: {avg_scores[2]:.4f}')
print(f'Average Recall: {avg_scores[3]:.4f}')